In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

In [ ]:
# ============ 누락 선수 수비이닝 채우기 — 2단계가 손대지 않는 나머지 (누적만) ============
# 누적(player_defense_{YEAR}.json)에 없는 선수는 2단계에서 prev 가 비어 delta 가 '시즌 전체
# 누적'이 되고, 그 값이 경기 하나에 통째로 박힌다. 그 오염 자체는 2단계 셀 안의
# '[베이스라인]' 가드가 막는다 — 어차피 2단계가 그 선수를 긁으므로, 거기서 처리하는 편이
# 크롤 한 번을 아낀다(2026-08-27 기준 누락 야수 41명 중 38명이 2단계 대상과 겹쳤다).
#
# 그래서 이 셀이 맡는 건 2단계가 영영 손대지 않는 선수뿐이다 — 출전 경기가 이미 전부
# processed 로 넘어가 버린 선수들. 위 기준일에 3명이었고, SKIP_STAGE2_TARGETS=True 면
# 대상이 54명에서 4명으로 줄어든다.
#
# 무엇을 하지 않는가 — defense_innings_{YEAR}.xlsx(경기별)에는 한 줄도 쓰지 않는다.
#   KBSA 가 수비이닝을 여러 경기치 한꺼번에 갱신하는 탓에, 지금 긁어서 얻는 값은 어디까지나
#   '시즌 누적'이다. 그걸 경기별로 쪼갤 근거가 없으므로 아예 시도하지 않는다.
#   (그래서 이 셀을 돌리면 그 선수의 밀린 이닝은 경기별로는 영영 남지 않는다. 어차피
#    쪼갤 수 없는 값이라 실제로 잃는 것은 없지만, 알고 돌리는 것과 모르고 돌리는 것은 다르다.)
#
# 대상 (person_no 중복제거 후) — 앞의 것이 급한 순서:
#   1) season_full 에 수비 포지션이 찍혔는데 누적에 아예 없는 선수   <- 오염원
#   2) 누적은 있으나 수비아웃 0(예전 크롤 시점엔 기록이 없었음)인데 지금은 포지션이 찍힌 선수
#   3) failed_defense2_person_no.txt 에 남은 요청 실패 선수
#   4) SCOPE="all" 일 때만: season_full 에 등장하는 나머지 전원(순수 투수 포함)
#
#   * 키가 있으면 '완료'로 보고 건너뛴다 -> 중간에 끊겨도 다시 돌리면 이어서 간다.
#   * players_{YEAR}.xlsx 는 읽지 않는다. 마스터 3,842명 중 2,188명이 미크롤이지만
#     대부분 season_full 에 나오지도 않는 선수라, 전부 긁으면 2시간이 헛돈다.
import requests
from bs4 import BeautifulSoup
import re
import os
import json
import time
import random
import unicodedata
import pandas as pd


# ==================== 설정 ====================
BASE_DIR = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
DATA_DIR = unicodedata.normalize("NFC", os.path.join(BASE_DIR, "players_by_season"))
YEAR = 2026

SEASON_FULL_PARQUET = unicodedata.normalize("NFC", f"{BASE_DIR}/season_full/season_full_{YEAR}.parquet")
SEASON_FULL_XLSX = unicodedata.normalize("NFC", f"{BASE_DIR}/season_full/season_full_{YEAR}.xlsx")
CUM_FILE = unicodedata.normalize("NFC", os.path.join(DATA_DIR, f"player_defense_{YEAR}.json"))
FAILED_FILE = os.path.join(DATA_DIR, "failed_defense2_person_no.txt")

# "gap" = 위 1~3번만(권장, 수십 명 규모) / "all" = season_full 전원(수백 명, 순수 투수 포함)
#   순수 투수는 페이지에 야수 포지션이 없어 {} 로 기록된다. 그래도 키가 생기므로, 나중에
#   그 선수가 외야를 뛰었을 때 2단계가 시즌 누적을 통째로 박는 사고는 막힌다.
SCOPE = "gap"

# 2단계가 어차피 크롤할 선수(= 아직 processed 가 아닌 경기에 출전한 선수)는 빼고 간다.
# 2단계에 '누적에 없던 선수는 베이스라인만' 가드가 들어 있어 그쪽에서 공짜로 처리된다.
# False 로 두면 여기서 먼저 다 채우지만, 그 대부분을 2단계가 몇 분 뒤 다시 긁는다.
SKIP_STAGE2_TARGETS = True
PROCESSED_FILE = unicodedata.normalize("NFC", os.path.join(DATA_DIR, f"defense_processed_games_{YEAR}.json"))

MAX_CRAWL = 0          # 0 = 제한 없음. 시간이 없으면 여기서 끊어도 다음 실행에 이어간다.
DRY_RUN = False        # True 면 대상 명단만 뽑고 크롤은 하지 않는다.

MIN_DELAY, MAX_DELAY, MAX_RETRY, SAVE_EVERY = 2.0, 5.0, 3, 50
VIEW_URL = "https://www.korea-baseball.com/info/player/player_view?person_no={}&gubun=P"

FIELD_POS = "포수|1루수|2루수|3루수|유격수|좌익수|중견수|우익수"
DEF_RE = re.compile(rf"({FIELD_POS})\(([\d.]+)\)")


# ==================== 이닝 <-> 아웃 (소수점=아웃, 0/1/2 뿐) ====================
def innings_to_outs(s):
    s = str(s).strip()
    if not s or s.lower() == "nan":
        return 0
    if "." in s:
        whole, frac = s.split(".", 1)
        return int(whole) * 3 + int(frac[:1] or 0)
    return int(s) * 3


def outs_to_innings(outs):
    outs = int(outs)
    return f"{outs // 3}.{outs % 3}"


# ==================== 로드 / 파싱 ====================
def _load_season_full_for_fill():
    for path, reader in [(SEASON_FULL_PARQUET, pd.read_parquet), (SEASON_FULL_XLSX, pd.read_excel)]:
        if os.path.exists(path):
            return reader(path)
    raise FileNotFoundError(f"season_full 없음: {SEASON_FULL_PARQUET} / {SEASON_FULL_XLSX}")


def parse_defense_fill(person_no, session):
    """gubun=P 프로필에서 그 시즌 포지션별 누적 수비이닝 dict. 기록 없으면 {}. 요청 실패면 None."""
    try:
        res = session.get(VIEW_URL.format(person_no), timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  [{person_no}] 요청 실패: {e}")
        return None
    res.encoding = "utf-8"
    try:
        soup = BeautifulSoup(res.text, "html.parser")
        result = {}
        for h4 in soup.find_all("h4"):
            text = h4.get_text(" ", strip=True)
            if str(YEAR) not in text:
                continue
            for pos, inn in DEF_RE.findall(text):
                result[pos] = inn
        return result
    except Exception as e:
        print(f"  [{person_no}] 파싱 오류: {e}")
        return {}


# ==================== 메인 ====================
def fill_missing_defense():
    sf = _load_season_full_for_fill()
    if "batter_person_no" not in sf.columns:
        print(f"season_full 에 batter_person_no 가 없습니다. 현재: {list(sf.columns)[:20]}")
        return

    # ---- season_full 에서 person_no 중복제거 ----
    def _ints(col):
        if col not in sf.columns:
            return set()
        return set(pd.to_numeric(sf[col], errors="coerce").dropna().astype("int64"))

    batters, pitchers = _ints("batter_person_no"), _ints("pitcher_person_no")
    everyone = batters | pitchers

    # 수비 포지션이 실제로 찍힌 선수만 급한 대상으로 본다. 지명타자·대타만 뛴 선수는
    # 수비이닝이 원래 0이라, 긁어도 {} 가 나오고 급할 이유도 없다.
    pos_players = set()
    if "position" in sf.columns:
        b = sf[sf["batter_person_no"].notna()].copy()
        b["_p"] = pd.to_numeric(b["batter_person_no"], errors="coerce")
        b = b[b["position"].astype(str).str.contains(FIELD_POS, regex=True, na=False)]
        pos_players = set(b["_p"].dropna().astype("int64"))

    # ---- 누적 로드 ----
    cum = {}
    if os.path.exists(CUM_FILE):
        with open(CUM_FILE, encoding="utf-8") as f:
            cum = json.load(f)
    have = {int(k) for k in cum}
    zero = {int(k) for k, v in cum.items()
            if sum(int(x.get("outs", 0)) for x in v.values()) == 0}

    # ---- 지난 실패 목록 ----
    failed_prev = []
    if os.path.exists(FAILED_FILE):
        with open(FAILED_FILE, encoding="utf-8") as f:
            failed_prev = [int(x) for x in f.read().split() if x.strip().isdigit()]

    # ---- 대상 만들기 (순서 유지 중복제거) ----
    g1 = sorted(pos_players - have)                      # 누적에 아예 없는 야수  <- 오염원
    g2 = sorted((pos_players & zero) - set(g1))          # 0으로 남아 있는 야수
    g3 = sorted(set(failed_prev) - set(g1) - set(g2))
    g4 = sorted(everyone - have - set(g1)) if SCOPE == "all" else []
    targets = list(dict.fromkeys(g1 + g2 + g3 + g4))

    # 2단계가 처리할 선수 빼기 — 아직 processed 가 아닌 경기에 한 번이라도 나오면 그쪽 몫이다.
    stage2 = set()
    if SKIP_STAGE2_TARGETS and os.path.exists(PROCESSED_FILE):
        with open(PROCESSED_FILE, encoding="utf-8") as f:
            done_games = {int(g) for g in json.load(f).get("game_ids", [])}
        _g = pd.to_numeric(sf["game_id"], errors="coerce")
        pend = sf[_g.notna() & ~_g.isin(done_games)]
        for _c in ("batter_person_no", "pitcher_person_no"):
            if _c in pend.columns:
                stage2 |= set(pd.to_numeric(pend[_c], errors="coerce").dropna().astype("int64"))
        before = len(targets)
        targets = [p for p in targets if p not in stage2]
        print(f"2단계가 맡을 {before - len(targets)}명 제외 (미처리 경기 출전자 {len(stage2)}명 중)")

    print(f"season_full 등장 {len(everyone)}명 (타자 {len(batters)} / 투수 {len(pitchers)})"
          f" · 수비 포지션 찍힘 {len(pos_players)}명 · 누적 보유 {len(have)}명")
    print(f"후보 {len(g1) + len(g2) + len(g3) + len(g4)}명 = 누락 야수 {len(g1)}"
          f" + 0으로 남은 야수 {len(g2)} + 지난 실패 {len(g3)}"
          + (f" + 나머지 전원 {len(g4)}" if g4 else ""))
    print(f"최종 대상 {len(targets)}명{': ' + str(targets[:10]) if targets else ''}"
          f"{' ...' if len(targets) > 10 else ''}")
    if not targets:
        print("채울 것이 없습니다 — 남은 구멍은 2단계가 처리합니다.")
        return
    if MAX_CRAWL and len(targets) > MAX_CRAWL:
        print(f"  이번엔 {MAX_CRAWL}명만 처리합니다 (나머지는 다음 실행에 이어서).")
        targets = targets[:MAX_CRAWL]
    print(f"예상 소요 약 {len(targets) * (MIN_DELAY + MAX_DELAY) / 2 / 60:.0f}분")
    if DRY_RUN:
        print("DRY_RUN - 크롤하지 않고 여기서 멈춥니다.")
        return

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})
    try:
        session.get("https://www.korea-baseball.com/info/player/player_list", timeout=10)
    except Exception:
        pass

    def _save():
        # 임시 파일에 쓰고 원자적으로 교체한다 — 중간에 끊겨도 반쪽짜리 JSON 이 남지 않게.
        tmp = CUM_FILE + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump({k: cum[k] for k in sorted(cum, key=int)}, f, ensure_ascii=False, indent=1)
        os.replace(tmp, CUM_FILE)

    filled, empty, failed = 0, 0, []
    for i, pno in enumerate(targets, 1):
        res, elapsed = None, 0.0
        for attempt in range(1, MAX_RETRY + 1):
            start = time.time()
            res = parse_defense_fill(pno, session)
            elapsed = time.time() - start
            if res is not None:
                break
            if attempt < MAX_RETRY:
                time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
        if res is None:
            failed.append(pno)
        else:
            entry = {}
            for pos, inn in res.items():
                o = innings_to_outs(inn)
                entry[pos] = {"outs": o, "innings": outs_to_innings(o)}
            cum[str(pno)] = entry
            if entry:
                filled += 1
            else:
                empty += 1
        if i % 20 == 0:
            print(f"  ...{i}/{len(targets)} (기록 {filled} / 없음 {empty} / 실패 {len(failed)})")
        if SAVE_EVERY and i % SAVE_EVERY == 0:
            _save()
        time.sleep(max(0, random.uniform(MIN_DELAY, MAX_DELAY) - elapsed))

    _save()

    # 실패 로그는 '아직 실패인 것'만 남긴다 — 예전에는 append 만 해서, 나중에 성공한
    # person_no 까지 계속 쌓여 다음 실행 대상을 부풀렸다.
    still = sorted(set(failed) | (set(failed_prev) - {int(k) for k in cum}))
    if still:
        with open(FAILED_FILE, "w", encoding="utf-8") as f:
            f.write("\n".join(str(x) for x in still) + "\n")
    elif os.path.exists(FAILED_FILE):
        os.remove(FAILED_FILE)

    print("\n" + "=" * 60)
    print(f"완료! 대상 {len(targets)}명 / 수비기록 있음 {filled} / 없음(투수·지명타자 등) {empty} / 실패 {len(failed)}")
    print(f"누적 보유 {len(have)} -> {len({int(k) for k in cum})}명")
    print(f"저장: {CUM_FILE}  (경기별 파일은 건드리지 않았습니다)")
    if failed:
        print(f"실패 person_no: {failed} (-> {FAILED_FILE}, 다음 실행에 자동 재시도)")
    print("=" * 60)


# ==================== 실행 ====================
fill_missing_defense()


In [ ]:
# ==================== 경기별 수비이닝 생성 (2단계) → defense_innings 롱포맷 ====================
# season_full로 새 game_id 출전 선수만 gubun=P 재크롤 → 이전 누적(player_defense_2026.xlsx)과
# 아웃 단위로 빼서(이닝 차) 그 경기 수비이닝을 defense_innings_2026.xlsx 에 append한다.
# 누적(이전/새)은 모두 player_defense_2026.xlsx(롱: person_no·position·outs·innings)로 관리하고,
# person_no로만 매칭한다(players_2026은 건드리지 않음).
#   ※ 소수점=아웃(0/1/2), 뺄셈은 무조건 아웃. 첫 실행은 베이스라인만 기록(과거는 못 쪼갬).
#     더블헤더 없음(고교) 전제, 애매하면 옵션 A(마지막 경기 합산 + flag='합산추정').
#   ※ 한 번에 다 돌리지 않는다 — 새로 추가된 경기부터 처리하고 밀린 경기는 다음 실행으로
#     미룬다(선수가 겹치는 경기는 통째로 같이 간다). 크롤 전에 표본 몇 명을 먼저 떠서
#     홈페이지가 아직 반영을 안 했으면 그 자리에서 접는다. 자세한 건 아래 설정 블록.
import requests
from bs4 import BeautifulSoup
import re
import os
import json
import time
import random
import unicodedata
import pandas as pd


# ==================== 설정 ====================
BASE_DIR = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"
DATA_DIR = unicodedata.normalize("NFC", os.path.join(BASE_DIR, "players_by_season"))
YEAR = 2026

# season_full 경로 — 필요시 실제 Drive 경로에 맞게 수정
SEASON_FULL_PARQUET = unicodedata.normalize("NFC", f"{BASE_DIR}/season_full/season_full_{YEAR}.parquet")
SEASON_FULL_XLSX = unicodedata.normalize("NFC", f"{BASE_DIR}/season_full/season_full_{YEAR}.xlsx")

PLAYER_FILE = unicodedata.normalize("NFC", os.path.join(DATA_DIR, f"players_{YEAR}.xlsx"))
CUM_FILE = unicodedata.normalize("NFC", os.path.join(DATA_DIR, f"player_defense_{YEAR}.json"))          # 누적(롱)
DEFENSE_FILE = unicodedata.normalize("NFC", os.path.join(DATA_DIR, f"defense_innings_{YEAR}.xlsx"))     # 경기별(롱)
PROCESSED_FILE = unicodedata.normalize("NFC", os.path.join(DATA_DIR, f"defense_processed_games_{YEAR}.json"))
FAILED_FILE = os.path.join(DATA_DIR, "failed_defense2_person_no.txt")
PROGRESS_FILE = unicodedata.normalize("NFC", os.path.join(DATA_DIR, f"defense_progress_{YEAR}.json"))  # 중단 지점

# 이미 processed(베이스라인)된 경기를 다시 잡고 싶을 때 game_id 나열. 예: [39057]
# ※ player_defense_{YEAR}.json 이 그 경기 "이전" 상태여야 이닝 차가 나온다(그 경기 크롤 전 누적).
REDO_GAMES = []

# ── 한 번에 얼마나 돌릴지 ────────────────────────────────────────────
# 홈페이지가 수비이닝을 늦게 올리는 탓에, 새 경기가 며칠씩 '미갱신'으로 밀리다가 한꺼번에
# 쌓인다. 실제로 22경기 493명을 29분 크롤하고 기록행 0으로 끝난 적이 있다(전부 미갱신).
# 아래 두 장치로 그 헛걸음을 줄인다.
PROCESS_BACKLOG = False   # False 면 새로 추가된 경기부터 처리하고, 밀린 경기는 다음 실행으로
PROBE_PER_GAME  = 2       # 경기마다 이만큼 먼저 떠본다. 전원 변화 없으면 '사이트 미반영'으로 중단 (0이면 끄기)
PROBE_MIN, PROBE_MAX = 6, 40   # 표본 수 하한/상한
SAVE_EVERY = 50           # 이만큼 크롤할 때마다 중간 저장 (중간에 끊겨도 다시 안 긁게)

# ── 한 번에 몇 명까지만 긁을지 ────────────────────────────────────
# 밀린 경기가 쌓이면 한 배치가 1,600명(=1시간 반)이 되기도 한다. 지금 당장 다른 일을
# 해야 하면(예: 경기 일정 먼저 올리기) 여기서 끊어 두고 나중에 이어서 돌리면 된다.
# 끊긴 지점은 defense_progress_{YEAR}.json 에 남아 다음 실행이 이어받고, 그 경기들은
# processed 로 넘기지 않으므로 완주하기 전까지 계속 대기 상태로 남는다.
#   0 = 제한 없음(끝까지)  /  예: 300 이면 300명만 긁고 저장한 뒤 종료
MAX_CRAWL_PER_RUN = 0

# ── 여러 경기에 걸친 이닝 차를 어떻게 할 것인가 ───────────────────
# 이 크롤은 '시즌 누적의 차이'를 읽는다. 그래서 한 선수가 미처리 경기 여러 개를 뛰었으면
# 그 차이가 어느 경기 몫인지 가릴 수 없다. 예전에는 마지막 경기에 통째로 몰아넣고
# flag='합산추정' 을 붙였는데, KBSA 가 여러 경기치를 한꺼번에 갱신하는 탓에 밀린 경기가
# 쌓이면 그 행이 절반 가까이 된다(2026-08-27 백로그 112경기 기준 466행 42%, 한 행에 최대
# 7경기치 = 63이닝). 못 믿을 값을 파일에 남기면 결국 누가 그냥 쓰게 되므로 기본은 버린다.
#   ※ 누적(player_defense_{YEAR}.json)은 사이트 값을 그대로 받으므로 이 설정과 무관하게
#     정확하다. 웹의 주 포지션 판정이 쓰는 것도 그쪽이다. 버려지는 건 경기별 상세뿐이다.
WRITE_SUM_ESTIMATE = False

VIEW_URL = "https://www.korea-baseball.com/info/player/player_view?person_no={}&gubun=P"
MIN_DELAY, MAX_DELAY, MAX_RETRY = 2.0, 5.0, 3

FIELD_POS = "포수|1루수|2루수|3루수|유격수|좌익수|중견수|우익수"
DEF_RE = re.compile(rf"({FIELD_POS})\(([\d.]+)\)")
POS_TOKEN_RE = re.compile(rf"({FIELD_POS})")
OUT_COLS = ["game_id", "person_no", "team", "player_name", "등번호", "position", "outs", "innings", "flag"]


# ==================== 이닝 <-> 아웃 (소수점=아웃, 0/1/2 뿐) ====================
def innings_to_outs(s):
    s = str(s).strip()
    if not s or s.lower() == "nan":
        return 0
    if "." in s:
        whole, frac = s.split(".", 1)
        return int(whole) * 3 + int(frac[:1] or 0)
    return int(s) * 3


def outs_to_innings(outs):
    outs = int(outs)
    return f"{outs // 3}.{outs % 3}"


# ==================== 파싱 / 로드 ====================
def parse_defense(person_no, session):
    """반환 dict(예 {"3루수":"181"}) / 요청 실패면 None."""
    try:
        res = session.get(VIEW_URL.format(person_no), timeout=10)
        res.raise_for_status()
    except Exception as e:
        print(f"  [{person_no}] 요청 실패: {e}")
        return None
    res.encoding = "utf-8"
    try:
        soup = BeautifulSoup(res.text, "html.parser")
        result = {}
        for h4 in soup.find_all("h4"):
            text = h4.get_text(" ", strip=True)
            if str(YEAR) not in text:
                continue
            for pos, inn in DEF_RE.findall(text):
                result[pos] = inn
        return result
    except Exception as e:
        print(f"  [{person_no}] 파싱 오류: {e}")
        return {}


def _log_failed(pno):
    with open(FAILED_FILE, "a", encoding="utf-8") as f:
        f.write(f"{pno}\n")


def _load_season_full():
    for path, reader in [(SEASON_FULL_PARQUET, pd.read_parquet), (SEASON_FULL_XLSX, pd.read_excel)]:
        if os.path.exists(path):
            return reader(path)
    raise FileNotFoundError(f"season_full 없음: {SEASON_FULL_PARQUET} / {SEASON_FULL_XLSX}")


def _load_cum():
    """player_defense_{year}.json -> {person_no(int): {position: outs}}"""
    if not os.path.exists(CUM_FILE):
        return {}
    with open(CUM_FILE, encoding="utf-8") as f:
        raw = json.load(f)
    return {int(k): {pos: int(v["outs"]) for pos, v in d.items()} for k, d in raw.items()}


# ==================== 실행 범위 고르기 ====================
def _load_not_updated():
    """지난 실행에서 '홈페이지 미반영'으로 남겨 둔 경기들."""
    p = os.path.join(DATA_DIR, f"defense_not_updated_{YEAR}.json")
    if not os.path.exists(p):
        return set()
    with open(p, encoding="utf-8") as f:
        return set(int(g) for g in json.load(f).get("game_ids", []))


def _components(games, players_of):
    """선수를 공유하는 경기끼리 묶는다.

    이 크롤은 선수 페이지의 '시즌 누적'을 읽어 이전 누적과 빼는 방식이라, 한 선수가
    대기 중인 경기 여러 개에 나오면 그 차이를 경기별로 쪼갤 수 없다. 그래서 경기를
    골라 처리할 때는 선수가 겹치는 경기를 통째로 같이 가져가야 한다 — 일부만 처리하면
    다른 경기 몫까지 이번 경기에 붙어 조용히 틀린다."""
    parent = {g: g for g in games}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    by_player = {}
    for g in games:
        for pno in players_of.get(g, ()):
            by_player.setdefault(pno, []).append(g)
    for gs in by_player.values():
        for g in gs[1:]:
            union(gs[0], g)

    groups = {}
    for g in games:
        groups.setdefault(find(g), set()).add(g)
    return list(groups.values())


def _pick_probe(batch, players_of, pos_of, skip=()):
    """사이트 반영 여부를 떠보기 위한 표본 선수.

    경기마다 '그 경기에서 수비 포지션이 확인된' 선수를 우선 뽑는다 — 지명타자나 대타만
    뽑으면 수비이닝이 원래 0이라 미반영과 구분이 안 된다."""
    # skip: 지난 실행에서 이미 긁은 선수. 이들은 누적이 최신이라 무조건 delta 0 이고,
    # 표본에 섞이면 '사이트 미반영'으로 오판하게 만든다.
    cands = {}
    for g in sorted(batch):
        pool = [p for p in sorted(players_of.get(g, ())) if p not in skip]
        c = [p for p in pool if pos_of.get((p, g))]   # 수비 포지션이 찍힌 선수
        cands[g] = c or pool

    # 경기를 돌아가며 한 명씩 뽑는다. 한 경기씩 몰아 뽑으면 PROBE_MAX 에 걸렸을 때 뒤쪽
    # 경기가 표본에서 통째로 빠져, 그 경기만 갱신됐을 때 '미반영'으로 잘못 접는다.
    picked, seen = [], set()
    for k in range(PROBE_PER_GAME):
        for g in sorted(batch):
            if len(picked) >= PROBE_MAX:
                return picked
            if k < len(cands[g]) and cands[g][k] not in seen:
                seen.add(cands[g][k])
                picked.append(cands[g][k])
    return picked


def _write_not_updated(game_ids):
    """다음 실행에 다시 볼 경기 목록(없으면 파일 정리)."""
    p = os.path.join(DATA_DIR, f"defense_not_updated_{YEAR}.json")
    if game_ids:
        with open(p, "w", encoding="utf-8") as f:
            json.dump({"game_ids": sorted(game_ids)}, f, ensure_ascii=False)
    elif os.path.exists(p):
        os.remove(p)

# ==================== 메인 ====================
def build_defense_game_log():
    sf = _load_season_full()
    if not {"game_id", "batter_person_no"}.issubset(sf.columns):
        print(f"season_full에 game_id·batter_person_no 필요. 현재: {list(sf.columns)[:20]}")
        return
    if not os.path.exists(CUM_FILE):
        print(f"{CUM_FILE} 없음 — 1단계(초기 적재) 먼저 실행하세요.")
        return

    cum = _load_cum()  # 이전 누적 {pno: {pos: outs}}
    # players 마스터: person_no -> (성명, 등번호)
    pmap = {}
    if os.path.exists(PLAYER_FILE):
        _pl = pd.read_excel(PLAYER_FILE, engine="openpyxl")
        if {"person_no", "성명"}.issubset(_pl.columns):
            for _, _r in _pl.dropna(subset=["person_no"]).iterrows():
                pmap[int(_r["person_no"])] = (_r.get("성명"), _r.get("등번호"))
    current_games = set(pd.to_numeric(sf["game_id"], errors="coerce").dropna().astype(int))

    processed = set()
    if os.path.exists(PROCESSED_FILE):
        with open(PROCESSED_FILE, encoding="utf-8") as f:
            processed = set(int(g) for g in json.load(f).get("game_ids", []))

    # 첫 실행: 베이스라인만 기록(과거 경기는 경기별로 못 쪼갬)
    if REDO_GAMES:
        _redo = {int(g) for g in REDO_GAMES}
        processed -= _redo
        print(f"재처리 대상으로 제외: {sorted(_redo)}")

    if not processed:
        with open(PROCESSED_FILE, "w", encoding="utf-8") as f:
            json.dump({"game_ids": sorted(current_games)}, f, ensure_ascii=False)
        print(f"베이스라인 설정: 현재 {len(current_games)}경기를 '누적 반영됨'으로 기록.")
        print("→ 이후 새 경기가 추가되면 이 셀을 다시 돌려서 경기별 수비이닝을 남깁니다.")
        return

    pending = current_games - processed
    if not pending:
        print("새 경기 없음 — 경기별 수비이닝 갱신할 것이 없습니다.")
        return
    stale = _load_not_updated() & pending
    fresh = pending - stale
    print(f"대기 {len(pending)}경기 (새로 추가 {len(fresh)} / 지난번 홈페이지 미반영 {len(stale)})")

    sf2 = sf.copy()
    sf2["_gid"] = pd.to_numeric(sf2["game_id"], errors="coerce")
    innew = sf2[sf2["_gid"].isin(pending)]

    # 선수별 '시즌 전체' 출전 경기. 누적에 없던 선수를 만났을 때, 그 시즌 누적을 이번
    # 경기에 귀속해도 되는지 판단하는 데만 쓴다 — 시즌 통틀어 1경기뿐이면 누적 = 그 경기다.
    season_games_of = {}
    for _col in ("batter_person_no", "pitcher_person_no"):
        if _col not in sf2.columns:
            continue
        _t = sf2[sf2[_col].notna()]
        for _p, _g in zip(pd.to_numeric(_t[_col], errors="coerce"), _t["_gid"]):
            if pd.notna(_p) and pd.notna(_g):
                season_games_of.setdefault(int(_p), set()).add(int(_g))

    # 새 경기 출전 선수 = 타자 ∪ 투수 (투수도 외야 등 수비 포지션을 뛸 수 있어 함께 확인)
    games_of, team_of, pos_of = {}, {}, {}
    has_pos = "position" in innew.columns
    for _, r in innew[innew["batter_person_no"].notna()].iterrows():
        pno = int(float(r["batter_person_no"])); g = int(r["_gid"])
        games_of.setdefault(pno, set()).add(g)
        key = (pno, g)
        if key not in team_of and pd.notna(r.get("team")):
            team_of[key] = r["team"]
        if has_pos:
            pos_of.setdefault(key, set()).update(POS_TOKEN_RE.findall(str(r.get("position", ""))))
    if "pitcher_person_no" in innew.columns:
        for _, r in innew[innew["pitcher_person_no"].notna()].iterrows():
            pno = int(float(r["pitcher_person_no"])); g = int(r["_gid"])
            games_of.setdefault(pno, set()).add(g)
            key = (pno, g)
            if key not in team_of:
                t = r.get("def_team") if pd.notna(r.get("def_team")) else r.get("team")
                if pd.notna(t):
                    team_of[key] = t
    players_of = {}
    for _p, _gs in games_of.items():
        for _g in _gs:
            players_of.setdefault(_g, set()).add(_p)

    # 새로 추가된 경기부터 처리하고 밀린 것은 다음으로 미룬다. 다만 선수가 겹치는 경기는
    # 통째로 같이 가져간다 — 시즌 누적의 차이를 쓰는 방식이라 겹친 경기를 쪼개 처리하면
    # 다른 경기 몫까지 이번 경기에 붙어 조용히 틀린다.
    # (새 경기가 하나도 없는 날에는 밀린 것을 전부 처리하므로 backlog 가 영영 남지 않는다.)
    if PROCESS_BACKLOG or not fresh:
        new_games, deferred = set(pending), set()
    else:
        new_games = set()
        for comp in _components(pending, players_of):
            if comp & fresh:
                new_games |= comp
        deferred = pending - new_games
    if deferred:
        print(f"  이번엔 {len(new_games)}경기만 처리하고 {len(deferred)}경기는 다음 실행으로 미룹니다: {sorted(deferred)}")
        print("   (한 번에 다 돌리려면 PROCESS_BACKLOG = True)")

    affected = sorted(p for p in games_of if games_of[p] & new_games)
    games_of = {p: games_of[p] & new_games for p in affected}

    # ── 중단된 실행 이어받기 ────────────────────────────────────────
    # 이 크롤은 '시즌 누적의 차이'를 읽는다. 한 번 긁은 선수는 누적이 이미 최신이라
    # 다시 긁어도 delta 가 0 이다. 그래서 중간에 끊고 다시 돌리면 (1) 이미 긁은 선수를
    # 헛되이 다시 긁고, 더 나쁘게는 (2) 표본이 정렬 기반이라 매번 같은 사람을 뽑는 탓에
    # '전원 변화 없음'으로 읽혀, 사이트가 멀쩡히 갱신됐는데도 통째로 중단된다.
    # (2026-08-27 실제로 발생 — 표본 40명이 전부 직전 중단분이라 몇 번을 돌려도 멈췄다.)
    # 그래서 어디까지 긁었는지를 파일로 남기고 다음 실행이 이어받는다.
    resumed, observed_seed = set(), {}
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, encoding="utf-8") as f:
            _pr = json.load(f)
        if set(int(g) for g in _pr.get("games", [])) <= new_games:
            resumed = {int(p) for p in _pr.get("done", [])} & set(affected)
            observed_seed = {int(k): int(v) for k, v in _pr.get("observed", {}).items()}
        else:
            print("  이어받기 파일이 지금 배치와 맞지 않아 무시합니다.")
    todo = [p for p in affected if p not in resumed]
    print(f"처리 대상 {len(new_games)}경기 / 선수 {len(affected)}명 재크롤 (타자+투수)"
          + (f" — 지난 실행에서 끝낸 {len(resumed)}명 건너뜀, 남은 {len(todo)}명" if resumed else ""))

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})
    try:
        session.get("https://www.korea-baseball.com/info/player/player_list", timeout=10)
    except Exception:
        pass

    rows, failed = [], []
    # observed: 경기별 '사이트가 반영했다'는 증거(아웃 수). 행을 버리든 말든 여기엔 남긴다.
    #   지난 실행 몫도 이어받는다 — 안 그러면 그 경기가 '미갱신'으로 오인된다.
    # crawled: 이번 배치에서 끝낸 선수(진행 파일에 기록)
    # dropped: 버린 합산추정 (마지막 요약용)
    observed, dropped = dict(observed_seed), []
    crawled = set(resumed)

    def _flush():
        """지금까지 잡은 것을 파일에 반영한다 — 중간에 끊겨도 다시 안 긁도록 자주 부른다.

        rows 와 cum 은 반드시 같이 저장한다. cum 만 앞서 나가면 그 선수의 이닝 차가 영영
        사라지고(다음 실행엔 delta 0), rows 만 앞서 나가면 다음 실행에 두 번 붙는다."""
        if rows:
            new_df = pd.DataFrame(rows, columns=OUT_COLS)
            if os.path.exists(DEFENSE_FILE):
                old_df = pd.read_excel(DEFENSE_FILE, engine="openpyxl")
                combined = pd.concat([old_df, new_df], ignore_index=True)
                combined = combined.drop_duplicates(subset=["game_id", "person_no", "position"], keep="last")
            else:
                combined = new_df
            combined.to_excel(DEFENSE_FILE, index=False, engine="openpyxl")
        cum_out = {str(pp): {pos: {"outs": int(o), "innings": outs_to_innings(o)} for pos, o in dd.items()}
                   for pp, dd in sorted(cum.items())}
        with open(CUM_FILE, "w", encoding="utf-8") as f:
            json.dump(cum_out, f, ensure_ascii=False, indent=1)
        # 어디까지 긁었는지 — cum 과 반드시 같이 저장한다(따로 놀면 이어받기가 어긋난다).
        with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
            json.dump({"games": sorted(new_games), "done": sorted(crawled),
                       "observed": {str(k): int(v) for k, v in observed.items()}},
                      f, ensure_ascii=False)

    def _crawl_one(pno):
        """한 선수를 긁어 이닝 차를 rows 에 넣는다. 반환: 이닝 차가 있었는지."""
        new_def = None
        elapsed = 0.0
        for attempt in range(1, MAX_RETRY + 1):
            start = time.time()
            new_def = parse_defense(pno, session)
            elapsed = time.time() - start
            if new_def is not None:
                break
            if attempt < MAX_RETRY:
                time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))
            else:
                failed.append(pno); _log_failed(pno)
        if new_def is None:
            return False
        crawled.add(pno)

        prev = cum.get(pno, {})                                   # {pos: outs}
        new_outs = {pos: innings_to_outs(inn) for pos, inn in new_def.items()}

        # 누적에 아예 없던 선수는 prev 가 비어 delta 가 '시즌 전체 누적'이 된다. 그대로 쓰면
        # 시즌 내내 쌓인 수비이닝이 이 경기 하나에 통째로 박힌다(2026-08-27 기준 41명이 해당).
        # 시즌 통틀어 이 경기 하나뿐인 선수만 귀속하고(그 경우 누적 = 그 경기라 정확하다),
        # 나머지는 베이스라인만 세우고 행을 만들지 않는다 — 쪼갤 근거가 없기 때문이다.
        # (그래서 이 선수들의 밀린 이닝은 경기별로는 남지 않는다. KBSA 가 여러 경기치를
        #  한꺼번에 갱신하는 이상 어차피 복원할 수 없는 값이다.)
        newcomer_flag = ""
        if pno not in cum:
            _sg = season_games_of.get(pno, set())
            if len(_sg) == 1 and len(games_of.get(pno, set())) == 1:
                newcomer_flag = "신규1경기"
            else:
                cum[pno] = new_outs
                print(f"  [베이스라인] {pno} 누적에 없던 선수 — 시즌 {len(_sg)}경기라 쪼갤 수 없어 행 생략")
                time.sleep(max(0, random.uniform(MIN_DELAY, MAX_DELAY) - elapsed))
                return False
        delta = {pos: new_outs[pos] - prev.get(pos, 0)
                 for pos in new_outs if new_outs[pos] - prev.get(pos, 0) > 0}

        pg = sorted(games_of.get(pno, set()))
        for pos, d in delta.items():
            flag = newcomer_flag
            if len(pg) == 1:
                assign = [(pg[0], d)]
            else:
                gwp = [g for g in pg if pos in pos_of.get((pno, g), set())]
                if len(gwp) == 1:
                    assign = [(gwp[0], d)]
                else:
                    # 옵션 A: 마지막(최근) 경기에 합산 + 플래그
                    target_g = (gwp[-1] if gwp else pg[-1])
                    assign = [(target_g, d)]
                    flag = "합산추정"
            for g, outs in assign:
                # 사이트가 이 경기를 반영했다는 증거는 행을 쓰든 버리든 남겨야 한다.
                # 안 그러면 합산추정만 나온 경기가 '미갱신'으로 오인돼 pending 에 계속
                # 남고, 그 사이 cum 은 이미 올라가 다음 실행엔 delta 0 이라 복구도 안 된다.
                observed[g] = observed.get(g, 0) + int(outs)
                if flag == "합산추정" and not WRITE_SUM_ESTIMATE:
                    dropped.append((pno, pos, int(outs), g, len(pg)))
                    continue
                _nm, _no = pmap.get(pno, ("", ""))
                rows.append({
                    "game_id": g, "person_no": pno, "team": team_of.get((pno, g), ""),
                    "player_name": _nm, "등번호": _no,
                    "position": pos, "outs": int(outs), "innings": outs_to_innings(outs), "flag": flag,
                })

        cum[pno] = new_outs                                       # 누적 갱신
        time.sleep(max(0, random.uniform(MIN_DELAY, MAX_DELAY) - elapsed))
        return bool(delta)

    # ── 먼저 떠보기: 홈페이지가 아직 수비이닝을 안 올렸으면 전원 크롤이 통째로 헛걸음이다.
    #    (실제로 22경기 493명을 29분 긁고 기록행 0으로 끝난 적이 있다.)
    #    경기마다 수비 포지션이 확인된 선수를 몇 명씩 떠서, 이닝 차가 하나도 없으면 접는다.
    # 이 배치의 경기에 이미 기록행이 있으면 사이트는 이미 반영한 것이다 — 직접 증거가
    # 있는데 표본을 다시 뜰 이유가 없고, 뜨면 오판만 부른다.
    _known_updated = False
    if os.path.exists(DEFENSE_FILE):
        try:
            _dg = pd.read_excel(DEFENSE_FILE, engine="openpyxl")["game_id"]
            _known_updated = bool(set(pd.to_numeric(_dg, errors="coerce").dropna().astype(int)) & new_games)
        except Exception:
            pass
    if _known_updated and PROBE_PER_GAME:
        print("  이 배치의 경기에 이미 기록행이 있어 표본 떠보기를 건너뜁니다 (사이트 반영 확인됨).")

    order = list(todo)
    if PROBE_PER_GAME and not _known_updated and len(todo) > PROBE_MIN:
        probe = [p for p in _pick_probe(new_games, players_of, pos_of, skip=resumed)
                 if p in games_of and p not in resumed]
        if len(probe) >= PROBE_MIN:
            print(f"  먼저 {len(probe)}명만 떠봅니다 (홈페이지 반영 여부 확인)")
            hit = False
            for j, pno in enumerate(probe, 1):
                if _crawl_one(pno):
                    hit = True
                if j % 10 == 0:
                    print(f"    ...{j}/{len(probe)}")
            if not hit:
                _flush()
                print(f"  → 표본 {len(probe)}명 전원 이닝 변화 없음. 홈페이지가 아직 이 경기들을 "
                      f"반영하지 않은 것으로 보고 {len(todo) - len(probe)}명 크롤을 건너뜁니다.")
                print(f"  대기 유지: {sorted(new_games)}")
                _write_not_updated(sorted(set(new_games) | deferred))
                print("=" * 60)
                print(f"중단(사이트 미반영) — 크롤 {len(probe)}명 / 실패 {len(failed)}")
                print("=" * 60)
                return
            print("  → 변화가 있어 나머지도 진행합니다.")
            order = [p for p in todo if p not in set(probe)]

    truncated = 0
    if MAX_CRAWL_PER_RUN and len(order) > MAX_CRAWL_PER_RUN:
        truncated = len(order) - MAX_CRAWL_PER_RUN
        order = order[:MAX_CRAWL_PER_RUN]
        print(f"  이번 실행은 {MAX_CRAWL_PER_RUN}명까지만 긁고 나머지 {truncated}명은 다음 실행으로 넘깁니다.")

    for i, pno in enumerate(order, 1):
        _crawl_one(pno)
        if i % 20 == 0:
            print(f"  ...{i}/{len(order)} 처리")
        if SAVE_EVERY and i % SAVE_EVERY == 0:
            _flush()
            print(f"    (중간 저장 {i}명까지)")

    # 도중에 끊었으면 여기서 마무리하지 않는다. 이 배치를 다 긁지 않았는데 경기를
    # processed 로 넘기면, 남은 선수들의 이닝 차가 영영 어느 경기에도 안 붙는다.
    if truncated:
        _flush()
        _write_not_updated(sorted(set(new_games) | deferred))
        print()
        print("=" * 60)
        print(f"이번 실행 몫만 끝냄 — 크롤 {len(order)}명 / 남은 {truncated}명"
              f" / 기록행 {len(rows)} / 실패 {len(failed)}")
        print(f"  {len(new_games)}경기는 대기 상태 그대로 둡니다. 이 셀을 다시 돌리면 이어서 갑니다.")
        print(f"  진행 상황: {PROGRESS_FILE}")
        print("=" * 60)
        return

    # 홈페이지 미갱신 감지: season_full엔 있으나(=새 경기) 사이트 수비이닝이 아직 반영 안 돼
    # 잡힌 수비아웃이 0인 경기는 '갱신 안됨'으로 두고 processed에 넣지 않는다 → 다음 실행에 자동 재처리.
    not_updated = sorted(g for g in new_games if observed.get(g, 0) == 0)
    ok_games = new_games - set(not_updated)
    if not_updated:
        print(f"  [갱신 안됨] 홈페이지 수비이닝 미반영 → 재처리 대상 유지: {not_updated}")

    new_df = pd.DataFrame(rows, columns=OUT_COLS)
    _flush()
    if new_df.empty:
        print("  기록할 새 수비이닝 없음 — 이번 실행엔 변화가 없었습니다.")
    processed |= ok_games
    with open(PROCESSED_FILE, "w", encoding="utf-8") as f:
        json.dump({"game_ids": sorted(processed)}, f, ensure_ascii=False)

    # 이 배치를 끝까지 돌았으니 이어받기 파일은 필요 없다.
    if os.path.exists(PROGRESS_FILE):
        os.remove(PROGRESS_FILE)

    # 다음 실행에 다시 볼 경기: 이번에 미반영이던 것 + 이번에 미룬 것
    _write_not_updated(sorted(set(not_updated) | deferred))

    print("\n" + "=" * 60)
    if dropped:
        _n = len(dropped)
        _o = sum(x[2] for x in dropped)
        print(f"  [합산추정 {'기록' if WRITE_SUM_ESTIMATE else '버림'}] {_n}행 / {_o}아웃"
              f" — 미처리 경기 여러 개에 걸쳐 어느 경기 몫인지 가릴 수 없는 값입니다.")
        for _p, _pos, _d, _g, _ng in sorted(dropped, key=lambda x: -x[4])[:5]:
            print(f"     예: {_p} {_pos} {_d}아웃 (밀린 {_ng}경기분, 마지막 {_g})")
        if not WRITE_SUM_ESTIMATE:
            print("     → 누적에는 반영됐고 경기별 파일에만 안 씁니다"
                  " (남기려면 WRITE_SUM_ESTIMATE = True).")
    print(f"완료! 처리 {len(new_games)}경기 (반영 {len(ok_games)} / 미갱신 {len(not_updated)})"
          f"{f' / 미룬 경기 {len(deferred)}' if deferred else ''}"
          f" / 대상 {len(affected)}명 / 기록행 {len(new_df)} / 실패 {len(failed)}")
    if not_updated:
        print(f"  미갱신(다음 실행에 재시도): {not_updated}")
    print(f"저장: {DEFENSE_FILE} (경기별) · {CUM_FILE} (누적 갱신)")
    if failed:
        print(f"실패 person_no: {failed} (→ {FAILED_FILE})")
    print("=" * 60)


# ==================== 실행 ====================
build_defense_game_log()